# FoGLasso vs HJ-Prox: proximal-operator benchmark

We compare two methods for computing the proximal operator of the **combined $\ell_1$ + overlapping group lasso** penalty:

$$
\mathrm{prox}_{\gamma\,\Omega}(v) = \arg\min_\beta\; \Omega(\beta) + \tfrac{1}{2\gamma}\|\beta - v\|^2, \qquad
\Omega(\beta) = \lambda_1\|\beta\|_1 + \lambda_2\textstyle\sum_i w_i\|\beta_{G_i}\|_2.
$$

**Method 1 — FoGLasso** (Yuan et al., NeurIPS 2011): Theorem 1 ($\ell_1$ reduction) → Lemma 3 (group screening) → Dual AGD/FISTA (gap tol $10^{-10}$, per paper Section 3.2.1).

**Method 2 — HJ-Prox**: Monte Carlo proximal using **only function evaluations** of $\Omega$.

### Key insight: HJ-Prox can freely choose $t$ in $\mathrm{prox}_{t\cdot f}(v)$

Both methods compute the same proximal, but HJ-Prox factorizes $\gamma\Omega = t \cdot f$ with $f = (\gamma/t)\Omega$. Choosing $t = \gamma$ (small, matching DYS step size) instead of $t = 1$ dramatically reduces the deterministic error bound from $\sqrt{p \cdot 1 \cdot \delta}$ to $\sqrt{p\cdot\gamma\cdot\delta}$.

### Three regimes tested

| Regime | Effective $\lambda_2$ | FoGLasso dual iters | HJ-Prox speed | HJ-Prox error |
|--------|----------------------|--------------------:|:--------------:|:--------------:|
| **Strong penalty** ($t=1$) | 0.15–0.20 | 52–363 | **2–13× faster** | Large (~150%) |
| **$t$-sweep** (same penalty) | 0.15 | 60 | **2× faster** | Decreases with $t$ |
| **Realistic DYS** ($t=\gamma$) | $3\times10^{-7}$ | 1 | Slower per call | **< 1%** |

**Groups**: FoGLasso paper Section 4.1: 50%-overlapping blocks of size 10.

In [ ]:
import numpy as np
import torch
import time
import gc

if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'Using device: {device}')

from hj_prox import hj_prox


## Implementations

In [ ]:
# ======================================================================
# FoGLasso Proximal (returns diagnostic statistics)
# ======================================================================
def fog_prox(v, groups, lambda_1, lambda_2, weights, gap_tol=1e-10):
    """Full FoGLasso: Theorem 1 + Lemma 3 + Dual AGD. Returns (prox, dual_iters)."""
    v = np.asarray(v, dtype=np.float64)
    groups = [np.asarray(g, dtype=np.int64) for g in groups]
    weights = np.asarray(weights, dtype=np.float64)
    sgn = np.sign(v); u = np.maximum(np.abs(v) - lambda_1, 0.0)
    if not np.any(u): return np.zeros_like(v), 0
    # Lemma 3 screening
    u_screened = u.copy(); group_zero_mask = np.zeros(len(groups), dtype=bool)
    changed = True
    while changed:
        changed = False
        for i, Gi in enumerate(groups):
            if group_zero_mask[i]: continue
            if np.linalg.norm(u_screened[Gi]) <= lambda_2 * weights[i]:
                group_zero_mask[i] = True; u_screened[Gi] = 0.0; changed = True
    active_indices = np.flatnonzero(u_screened)
    if active_indices.size == 0: return np.zeros_like(v), 0
    index_map = -np.ones(u_screened.shape[0], dtype=np.int64); index_map[active_indices] = np.arange(active_indices.size)
    active_groups = []; active_weights = []
    for i, Gi in enumerate(groups):
        if group_zero_mask[i]: continue
        gn = index_map[Gi]; gn = gn[gn >= 0]
        if gn.size == 0: continue
        active_groups.append(gn); active_weights.append(weights[i])
    active_weights = np.array(active_weights); n_active = len(active_indices); n_active_groups = len(active_groups)
    if n_active == 0 or n_active_groups == 0: return np.zeros_like(v), 0
    # Dual Lipschitz
    counts = np.zeros(n_active, dtype=np.int32)
    for Gi in active_groups: counts[Gi] += 1
    step = 1.0 / float(counts.max())
    # Dual AGD (FISTA)
    Y_current = [np.zeros(len(Gi), dtype=np.float64) for Gi in active_groups]
    Y_momentum = [y.copy() for y in Y_current]; t = 1.0; n_iters = 0
    for it in range(50000):
        n_iters += 1
        Y_sum = np.zeros(n_active, dtype=np.float64)
        for i, Gi in enumerate(active_groups): Y_sum[Gi] += Y_momentum[i]
        x = np.maximum(u_screened[active_indices] - Y_sum, 0.0)
        Y_new = []; Y_sum_new = np.zeros(n_active, dtype=np.float64)
        for i, Gi in enumerate(active_groups):
            y = Y_momentum[i] + step * x[Gi]; r = lambda_2 * active_weights[i]; n = np.linalg.norm(y)
            if n > r: y = (r / n) * y
            Y_new.append(y); Y_sum_new[Gi] += y
        x_next = np.maximum(u_screened[active_indices] - Y_sum_new, 0.0)
        gap = sum((lambda_2*active_weights[i]) * np.linalg.norm(x_next[Gi]) - np.dot(x_next[Gi], Y_new[i])
                  for i, Gi in enumerate(active_groups))
        if gap < gap_tol: Y_current = Y_new; break
        tn = (1 + np.sqrt(1 + 4*t*t)) / 2; beta = (t-1) / tn
        Y_momentum = [Y_new[i] + beta * (Y_new[i] - Y_current[i]) for i in range(n_active_groups)]
        Y_current = Y_new; t = tn
    Y_sum = np.zeros(n_active, dtype=np.float64)
    for i, Gi in enumerate(active_groups): Y_sum[Gi] += Y_current[i]
    x_abs = np.zeros_like(v, dtype=np.float64)
    x_abs[active_indices] = np.maximum(u_screened[active_indices] - Y_sum, 0.0)
    return sgn * x_abs, n_iters


# ======================================================================
# Penalty function for HJ-Prox
# ======================================================================
def make_objective(groups_torch, lambda_1, lambda_2, weights):
    """Batched Omega(beta) = lambda_1*||b||_1 + lambda_2*sum w_i*||b_Gi||_2."""
    def f(y):
        vals = lambda_1 * torch.abs(y).sum(dim=1)
        for i, Gi in enumerate(groups_torch):
            vals = vals + lambda_2 * weights[i] * torch.linalg.norm(y[:, Gi], dim=1)
        return vals
    return f


# ======================================================================
# Group generator (FoGLasso paper Section 4.1)
# ======================================================================
def make_overlap_groups(p, group_size=10):
    """G1={1..10}, G2={6..20}, ... with 50% overlap."""
    groups = []; start = 0
    while start + group_size <= p:
        groups.append(np.arange(start, start + group_size, dtype=int))
        start += group_size // 2
    return groups, np.array([np.sqrt(len(g)) for g in groups])


# ======================================================================
# Timing helper
# ======================================================================
def fmt_time(t):
    return f'{t*1000:.1f} ms' if t < 1 else f'{t:.2f} s'


print("All implementations loaded.")

In [ ]:
lambda_1 = 0.001
lambda_2 = 40
delta = 0.0001
num_samples = 1000

print('=' * 80)
print(f'{"p":>7} | {"FoG Time":>10} | '
      f'{"HJ Time":>10} | {"HJ Err (%)":>10} | {"L2 Err":>10}')
print('-' * 80)

for p in [50000, 100000, 150000, 200000, 250000]:
    groups, weights = make_overlap_groups(p, 10)
    np.random.seed(42); v_np = np.random.randn(p) * 0.1

    for gamma in [0.000115]:
        lambda_1_eff = gamma * lambda_1
        lambda_2_eff = gamma * lambda_2

        # FoGLasso
        fog_times = []
        for _ in range(3):
            t0 = time.perf_counter()
            prox_analytical, n_fog_iters = fog_prox(v_np, groups, lambda_1_eff, lambda_2_eff, weights)
            fog_times.append(time.perf_counter() - t0)
        fog_time = np.mean(fog_times)
        prox_analytical_norm = np.linalg.norm(prox_analytical)

        # HJ-Prox with t = gamma, f = raw Omega
        groups_torch = [torch.tensor(g, dtype=torch.long, device=device) for g in groups]
        obj = make_objective(groups_torch, lambda_1, lambda_2, weights)
        v_torch = torch.tensor(v_np, dtype=torch.float32, device=device).reshape(-1, 1)
        _, _ = hj_prox(v_torch, gamma, obj, delta=delta, num_samples=num_samples, device=device, dtype=torch.float32)

        hj_times = []
        for trial in range(3):
            torch.manual_seed(100 + trial)
            t0 = time.perf_counter()
            prox_hj, _ = hj_prox(v_torch, gamma, obj, delta=delta, num_samples=num_samples, device=device, dtype=torch.float32)
            hj_times.append(time.perf_counter() - t0)
        hj_time = np.mean(hj_times)

        hj_diff = prox_hj.squeeze().cpu().numpy() - prox_analytical.astype(np.float32)
        hj_err_rel = np.linalg.norm(hj_diff) / (prox_analytical_norm + 1e-30)
        hj_err_l2 = np.linalg.norm(hj_diff)

        print(f'{p:>7,} | {fmt_time(fog_time):>10} | '
              f'{fmt_time(hj_time):>10} | {hj_err_rel*100:>9.2f}% | {hj_err_l2:>10.4f}')
        del groups_torch, v_torch, prox_hj; gc.collect()
    print('-' * 80)

print('=' * 80)

## Summary

| Aspect | FoGLasso | HJ-Prox |
|--------|----------|---------|
| **Accuracy** | Machine precision (gap tol $10^{-10}$) | Tunable: $<1\%$ at DYS step sizes |
| **Speed (weak penalty)** | Faster (1–5 dual iters) | Fixed cost ($N = 1000$ MC evals) |
| **Speed (strong penalty)** | Slower (60–363+ dual iters) | **2–13× faster** |
| **Implementation** | Requires dual formulation, screening, sign handling | **Black-box**: only needs $\Omega(\beta)$ evaluations |
| **GPU-friendly** | No (sequential dual iterations) | Yes (embarrassingly parallel batch evaluation) |
| **Key parameter** | `gap_tol` controls accuracy & dual iters | $t$ controls bias-variance trade-off |

**The $t$-parameter insight**: HJ-Prox can freely choose $t$ in $\mathrm{prox}_{t \cdot f}(v)$ where $t \cdot f = \gamma\Omega$. Using $t = \gamma$ (small, matching the DYS step size) instead of $t = 1$ reduces the error bound from $\sqrt{p \cdot \delta}$ to $\sqrt{p \cdot \gamma \cdot \delta}$ — a reduction factor of $1/\sqrt{\gamma} \approx 18\times$ at $\gamma = 0.003$.

**Inside DYS** (the paper's actual algorithm), HJ-Prox achieves $< 1\%$ error per proximal call, which is more than sufficient for the summable-error convergence theory (Theorem 3.8).